# TideTrace data preparation

**SIH26143** &middot; NTRO oil spill attribution &middot; data stage

This notebook turns the Zenodo Sentinel-1 SAR oil spill dataset into 512 px
training tiles and pushes them to the Hugging Face Hub. Nothing lands on the
demo laptop.

**Settings:** Accelerator *None*, Internet *On*, Persistence *None*.

**Before running:** add your Hugging Face token under
*Add-ons > Secrets* with the label `HF_TOKEN`.

## Why it streams

The Zenodo image archives are 10 to 40 GB compressed and expand to far more.
Kaggle's disk cannot hold one expanded, so this extracts a small batch of
chips, tiles them, deletes them, and moves on. Peak disk stays a few GB
regardless of archive size.

Batches are deliberately large. A solid 7z has to decompress from the start
of a block to reach a member, so many small rounds cost far more than a few
big ones. 60 chips per round is the compromise between that and disk.


## 1. Environment


In [ ]:
!pip -q install huggingface_hub py7zr rasterio 2>&1 | tail -2
import os, sys, subprocess, json
from pathlib import Path

def sh(*args):
    print('$', ' '.join(args), flush=True)
    subprocess.run(list(args), check=True)

# No credential is required to run this notebook. The TideTrace repos on the
# Hub are public, so code downloads anonymously, and results leave through
# Kaggle's own kernel output. A token is used only if you have chosen to add
# one as a Secret, in which case results are ALSO mirrored to the Hub.
HAVE_HF_TOKEN = False
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    HAVE_HF_TOKEN = True
    print('HF token found: results will also be mirrored to the Hub')
except Exception:
    print('No HF_TOKEN secret. Downloads are anonymous and results leave via')
    print('kernel output. This is the normal path and nothing is missing.')


## 2. Pull the TideTrace source

The tiling logic, the 3-class relabelling and the mask georeferencing rule
all live in the package. Duplicating them in a notebook is how the two
silently drift apart.


In [ ]:
from huggingface_hub import snapshot_download

CODE_DIR = Path('/kaggle/working/tidetrace')
snapshot_download(repo_id='N-1ACE/tidetrace-oil-unet', repo_type='model',
                  local_dir=str(CODE_DIR), allow_patterns=['code/**'])
sys.path.insert(0, str(CODE_DIR / 'code'))

# Keep every path Kaggle-local. The package writes under data/ by default.
os.environ['TIDETRACE_DATA'] = '/kaggle/working/data'
os.environ['TIDETRACE_MODELS'] = '/kaggle/working/models'

from app import config, hub
print('TideTrace', config.VERSION, '| tile', config.TILE, '| overlap', config.TILE_OVERLAP)
print('dataset repo:', hub.DATASET_REPO)
print('model repo  :', hub.MODEL_REPO)


## 3. Choose the Zenodo archive

| record | archive | size | what it holds |
| --- | --- | --- | --- |
| `13761290` | `02_Test_images_and_ground_truth.7z` | 9.9 GB | 150 chips per class, with ground truth |
| `8346860` | `01_Train_Val_Oil_Spill_images.7z` | 40.7 GB | oil train and validation |
| `8253899` | `01_Train_Val_Lookalike_images.7z` | 23.0 GB | look-alike |
| `8253899` | `01_Train_Val_No_Oil_Images.7z` | 22.9 GB | oil free |

Part III is the default. It is the smallest, it carries ground truth for all
three classes, and it is the set the IoU table is supposed to be reported on.
Point `RECORD` and `ARCHIVE` at a bigger part when you have the session time.


In [ ]:
RECORD  = '13761290'
ARCHIVE = '02_Test_images_and_ground_truth.7z'

# The sea budget was 150 and one shared pool. The folders are walked
# Lookalike, No oil, Oil, and every look-alike chip is relabelled sea
# because its mask is empty, so look-alikes took all 150 before
# `Images/No oil` was ever opened. The first run trained on 400 oil
# tiles, 150 look-alike hard negatives and not one tile of ordinary
# open water, and the model it produced answered 'oil' for very
# nearly every pixel of every real scene. Each source is capped on
# its own now, so the walk order stops mattering.
MAX_PLAIN_SEA   = 700    # Images/No oil
MAX_HARD_NEG    = 300    # look-alike chips whose mask is empty
MAX_SCENE_NEG   = 500    # open water cut from the oil scenes
MAX_PER_CLASS   = {2: 400, 1: 300,
                   0: MAX_PLAIN_SEA + MAX_HARD_NEG + MAX_SCENE_NEG}
BATCH_SIZE    = 60                          # chips per extraction round

WORK  = Path('/kaggle/temp/tidetrace'); WORK.mkdir(parents=True, exist_ok=True)
TILES = Path('/kaggle/working/tiles');  TILES.mkdir(parents=True, exist_ok=True)

import time
import urllib.request, urllib.error
UA = {'User-Agent': 'TideTrace/1.0 (SIH26143 research)'}

def _open(url, method='GET', timeout=180):
    return urllib.request.urlopen(
        urllib.request.Request(url, headers=UA, method=method), timeout=timeout)

def resolve(record, archive, tries=4):
    # Returns (url, size). The URL is deterministic; the API is optional.
    # Zenodo's API 504s often enough that a run must not hang on it. The
    # download path is stable and documented, so build it directly and
    # treat the API purely as a source of the size and the licence.
    url = 'https://zenodo.org/records/%s/files/%s?download=1' % (record, archive)
    size = 0
    for i in range(tries):
        try:
            meta = json.loads(_open('https://zenodo.org/api/records/' + record).read())
            entry = next(f for f in meta['files'] if f['key'] == archive)
            size = int(entry['size'])
            print('licence:', meta['metadata'].get('license', {}))
            break
        except Exception as exc:
            print('  api attempt %d/%d failed: %s' % (i + 1, tries, exc), flush=True)
            time.sleep(5 * (i + 1))
    if not size:
        print('  Zenodo API unavailable; asking the file itself for its size')
        for i in range(tries):
            try:
                with _open(url, method='HEAD') as r:
                    size = int(r.headers.get('Content-Length') or 0)
                if size:
                    break
            except Exception as exc:
                print('  head attempt %d/%d failed: %s' % (i + 1, tries, exc), flush=True)
                time.sleep(5 * (i + 1))
    if not size:
        raise SystemExit('Zenodo is not reachable from this session. Re-run later.')
    return url, size

URL, SIZE = resolve(RECORD, ARCHIVE)
print('%s  %.2f GB' % (ARCHIVE, SIZE / 1e9))
print(URL)


## 4. Download

Resumable by HTTP range, because a 10 GB transfer that fails at 95 percent
and starts over is how you lose a session.


In [ ]:
ARCH = WORK / ARCHIVE

def download(url, dest, expected, attempts=6):
    # Range-resumed download that retries the connection, not the file.
    for attempt in range(attempts):
        have = dest.stat().st_size if dest.exists() else 0
        if have >= expected:
            print('complete: %.2f GB' % (have / 1e9)); return
        req = urllib.request.Request(url, headers=UA)
        mode = 'wb'
        if have:
            req.add_header('Range', 'bytes=%d-' % have); mode = 'ab'
            print('resuming at %.2f GB (attempt %d)' % (have / 1e9, attempt + 1))
        t0, done, last = time.time(), have, -1
        try:
            with urllib.request.urlopen(req, timeout=300) as r, dest.open(mode) as fh:
                while True:
                    block = r.read(1 << 22)
                    if not block: break
                    fh.write(block); done += len(block)
                    pct = int(100 * done / expected)
                    if pct != last and pct % 5 == 0:
                        rate = (done - have) / max(1e-9, time.time() - t0) / 1e6
                        print('  %3d%%  %.2f / %.2f GB  %.0f MB/s'
                              % (pct, done / 1e9, expected / 1e9, rate), flush=True)
                        last = pct
        except Exception as exc:
            print('  transfer broke: %s; retrying from where it stopped' % exc, flush=True)
            time.sleep(10 * (attempt + 1))
            continue
    have = dest.stat().st_size if dest.exists() else 0
    if have < expected:
        raise SystemExit('download incomplete: %.2f of %.2f GB' % (have/1e9, expected/1e9))

download(URL, ARCH, SIZE)
print('on disk: %.2f GB' % (ARCH.stat().st_size / 1e9))


## 5. Stream: extract a batch, tile it, delete it

`build_tile_index` does the work that matters and is shared with the local
code path: it pairs each Sigma0 chip with its mask, **copies the SAR affine
and CRS onto the mask** before anything spatial happens, relabels by source
folder into the 3-class encoding, and drops tiles with too little signal.


In [ ]:
import shutil
import py7zr
from app.ml import dataset as ds

with py7zr.SevenZipFile(str(ARCH), 'r') as z:
    names = [n for n in z.getnames() if n.lower().endswith(('.tif', '.tiff'))]
print('%d rasters in the archive' % len(names))

# Group by parent folder so images and their masks are extracted together.
by_dir = ds.group_archive_members(names)
print('%-40s %6s  %-10s %s' % ('folder', 'files', 'role', 'class'))
for d in sorted(by_dir):
    role = 'MASK' if ds.is_mask_path(d) else 'image'
    klass = '-' if role == 'MASK' else ds.class_for_folder(d)
    print('%-40s %6d  %-10s %s' % (d[:40], len(by_dir[d]), role, klass))

# Part III ships Images/<class> beside Mask/<class>. If the split below
# shows no MASK folders, the pairing will silently label everything sea.
image_dirs = [d for d in by_dir if not ds.is_mask_path(d)]
mask_dirs  = [d for d in by_dir if ds.is_mask_path(d)]
print('\n%d image folders, %d mask folders' % (len(image_dirs), len(mask_dirs)))
assert image_dirs, 'no image folders found; check the archive layout above'


In [ ]:
STAGE = WORK / 'stage'
counts = {0: 0, 1: 0, 2: 0}
rounds = 0
hard_neg = 0    # look-alike chips with an empty mask, kept as sea
scene_neg = 0   # open-water tiles cut from oil scenes, kept as sea
plain_sea = 0   # tiles from Images/No oil

# Only image folders drive the loop. For each batch of chips,
# select_batch_targets adds the matching members from the parallel
# Mask tree, so images and their labels are extracted together. That
# pairing is unit tested in tests/test_dataset_layout.py against this
# archive's real member listing.
for folder in ds.image_folders(by_dir):
    klass = ds.class_for_folder(folder)
    if counts[klass] >= MAX_PER_CLASS.get(klass, 0):
        continue
    todo = sorted({Path(m).stem for m in by_dir[folder]})
    for i in range(0, len(todo), BATCH_SIZE):
        if counts[klass] >= MAX_PER_CLASS.get(klass, 0):
            break
        batch = todo[i:i + BATCH_SIZE]
        targets = ds.select_batch_targets(by_dir, folder, batch)
        if STAGE.exists(): shutil.rmtree(STAGE)
        STAGE.mkdir(parents=True, exist_ok=True)
        with py7zr.SevenZipFile(str(ARCH), 'r') as z:
            z.extract(path=str(STAGE), targets=targets)
        pairs = ds.index_pairs(STAGE)
        if not pairs:
            continue
        matched = sum(1 for _i, m, _k in pairs if m is not None)
        if rounds == 0:
            print('   first round: %d chips, %d with masks' % (len(pairs), matched))
        if matched == 0:
            for img, m, k in pairs[:5]:
                print('   %s -> mask %s (class %s)' % (Path(img).name, m, k))
            raise SystemExit(
                'No image was paired with a mask. Tiling would label every '
                'chip as sea and the oil class would vanish without an error.')
        remaining = {k: max(0, MAX_PER_CLASS.get(k, 0) - v) for k, v in counts.items()}
        idx = ds.build_tile_index(
            pairs, TILES, limit_per_class=remaining,
            max_hard_negatives=max(0, MAX_HARD_NEG - hard_neg),
            max_scene_negatives=max(0, MAX_SCENE_NEG - scene_neg),
            max_plain_sea=max(0, MAX_PLAIN_SEA - plain_sea))
        hard_neg  += idx.get('hard_negative_tiles', 0)
        scene_neg += idx.get('scene_negative_tiles', 0)
        plain_sea += idx.get('plain_sea_tiles', 0)
        gained = sum(idx['counts'].values())
        for k, v in idx['counts'].items():
            counts[int(k)] += v
        rounds += 1
        if gained == 0:
            # A look-alike folder never raises counts[1], because every
            # one of its chips is reclassified to sea. So the outer
            # budget check can never retire it, and once the hard
            # negative ceiling is reached it would extract and tile
            # thousands more chips for nothing. A round that yields no
            # tiles means this folder is finished.
            print('   no tiles from this round; moving on')
            break
        print('round %2d  %-30s tiles so far %s' % (rounds, folder[-30:], counts), flush=True)

if STAGE.exists(): shutil.rmtree(STAGE)
print('\ntiles written:', counts)
print('  of the sea tiles: %d plain, %d hard negative, %d scene negative'
      % (plain_sea, hard_neg, scene_neg))
assert plain_sea > 0, (
    'no tiles came from Images/No oil; the model would never see ordinary open water, which is how the first checkpoint learned to answer oil everywhere')
share = counts[2] / max(1, sum(counts.values()))
print('  oil tile share: %.1f%%' % (100 * share))
assert counts[0] > counts[2], (
    'sea tiles must outnumber oil tiles; the first run had that '
    'ratio inverted and the model learned to answer oil everywhere')


## 6. Rebuild the index over everything

Each streaming round wrote its own `index.json`. One pass at the end gives a
single index with the dB normalisation statistics computed over the whole set,
which is what the checkpoint records and inference reads back.


In [ ]:
import numpy as np
files = sorted(TILES.glob('*.npz'))
stats = ds._dataset_stats(TILES, sample=200)
# Count what is ON DISK, not what the loop attempted. Those two
# disagreed: one run reported 1812 tiles under `counts` while only
# 1164 files existed, because tiles from different source folders
# were overwriting each other. The old line here assigned instead of
# incrementing, so it always read zero and hid the discrepancy.
per_class = {0: 0, 1: 0, 2: 0}
for f in files:
    per_class[int(f.stem.split('_')[-3])] += 1

# Sample the pixel share EVENLY across the sorted list. Taking the
# first 400 names biases it to whichever source folder sorts first,
# which since tiles are named by source folder means one class.
px = np.zeros(3, dtype=np.int64)
step = max(1, len(files) // 400)
for f in files[::step]:
    px += np.bincount(np.load(f)['label'].ravel(), minlength=3)

if per_class != {int(k): v for k, v in counts.items()}:
    print('NOTE: attempted %s but %s are on disk' % (counts, per_class))

index = {'tiles': len(files), 'counts': {str(k): v for k, v in per_class.items()},
         'counts_attempted': {str(k): v for k, v in counts.items()},
         'tile_size': config.TILE, 'overlap': config.TILE_OVERLAP,
         'normalisation': stats, 'source_record': RECORD, 'source_archive': ARCHIVE,
         'pixel_share': (px / max(px.sum(), 1)).round(5).tolist()}
(TILES / 'index.json').write_text(json.dumps(index, indent=2))
print(json.dumps(index, indent=2))
print('\ntotal size: %.2f GB' % (sum(f.stat().st_size for f in files) / 1e9))


## 7. Sanity check the labels

Always look at the data. A silent labelling bug is far cheaper to find here
than after a twelve hour training run.


In [ ]:
import matplotlib.pyplot as plt

picks = [f for f in files if np.load(f)['label'].max() == 2][:4]
if picks:
    fig, ax = plt.subplots(2, len(picks), figsize=(4*len(picks), 8))
    ax = np.atleast_2d(ax)
    for j, f in enumerate(picks):
        z = np.load(f)
        ax[0, j].imshow(z['image'][0], cmap='gray'); ax[0, j].set_title('VV dB')
        ax[1, j].imshow(z['label'], vmin=0, vmax=2); ax[1, j].set_title('0 sea / 1 look-alike / 2 oil')
        for a in ax[:, j]: a.axis('off')
    plt.tight_layout(); plt.show()
else:
    raise SystemExit('No tile contains the oil class. The folder mapping is '
                     'wrong; see the role/class table in step 5.')


## 8. Hand the tiles on

The tile folder sits under `/kaggle/working`, which is this kernel's
**output**. The training notebook declares this kernel as a source and reads
it straight from `/kaggle/input`, so the tiles never cross the public
internet a second time and never touch a laptop.

With an `HF_TOKEN` secret present they are mirrored to the Hub as well,
which is worth doing once so the dataset is citable.


In [ ]:
size_gb = sum(f.stat().st_size for f in TILES.glob('*.npz')) / 1e9
print('kernel output: %d tiles, %.2f GB at %s' % (len(files), size_gb, TILES))
assert size_gb < 18, 'tile set will not fit in the 20 GB kernel output'
# Fail the kernel rather than report a green run that delivered nothing.
assert counts.get(2, 0) > 0, 'no oil tiles produced; see the folder table in step 5'

if HAVE_HF_TOKEN:
    print(hub.push_tiles(TILES, commit_message='tiles from %s' % ARCHIVE))
else:
    print('No token, so no Hub mirror. The training notebook reads this')
    print('kernel output directly, which is all it needs.')

print('\nNext: run the tidetrace-train notebook on a GPU.')
